# step 3 — pooled union: compilation & inspection

compiles the union of System 1 (cosine similarity) and System 2 (BERTopic) candidates,
saves `_pooled_union.json`, and provides cells for reviewing the candidate set.

**why pooled evaluation?**

pooled evaluation is applied because no single filtering method can
guarantee completeness; using at least two systems with weakly correlated
recall failures reduces the risk of systematic omissions.



**union rationale:** the union (not intersection) maximises recall — both systems have
independent failure modes, and any document missed by one may be caught by the other.

| section | what it shows |
|---|---|
| §3 | summary counts by source system + year distribution |
| §4 | high-confidence docs found by both systems |
| §5 | docs found only by cosine similarity |
| §5a | **decision:** which System-1-only policies to include (interactive y/n triage, saved by title) |
| §6 | docs found only by BERTopic — inspect topic by topic |
| §6a | **decision:** which BERTopic topics to keep |
| §7 | noise pool spot-check (topic -1) |
| §8 | compile final policy set & save |
| §9 | export evaluation log (Markdown) |

In [3]:
import gzip
import json
import random
import pandas as pd
from pathlib import Path
from IPython.display import display

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', None)

# ── CONFIGURATION ─────────────────────────────────────────────────────────────

def _find_root(marker='CLAUDE.md'):
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).exists():
            return p
    raise FileNotFoundError('project root not found — run from within the project directory')

PROJECT_ROOT      = _find_root()
OUTPUT_PREFIX     = 'germany_2016-2018'
BASE              = PROJECT_ROOT / 'data' / 'processed' / 'germany' / 'pre2023'

S1_FILE           = BASE / f'{OUTPUT_PREFIX}_cosinesim_final.json' #selection from system 1, cosine similarity
S2_FILE           = BASE / f'{OUTPUT_PREFIX}_berttopic_candidates.json' #policies that are in social policy relevant topics (from triage)
NOISE_FILE        = BASE / f'{OUTPUT_PREFIX}_berttopic_noise_pool.json' #policies that could not be matched to any topic  at all
TOPIC_ASSIGN_FILE = BASE / f'{OUTPUT_PREFIX}_berttopic_topic_assignments.json.gz' #all policies with their assigned topics
TOPIC_INFO_FILE   = BASE / f'{OUTPUT_PREFIX}_berttopic_topic_info.json'
OUT_FILE          = BASE / f'{OUTPUT_PREFIX}_pooled_union.json'
S1_DECISIONS_FILE = BASE / f'{OUTPUT_PREFIX}_s1_only_decisions.json' #§5a triage decisions for System-1-only policies, keyed by title

## §1. Load data

In [4]:
s1_docs    = json.loads(S1_FILE.read_text(encoding='utf-8'))
s2_docs    = json.loads(S2_FILE.read_text(encoding='utf-8'))
noise_docs = json.loads(NOISE_FILE.read_text(encoding='utf-8')) if NOISE_FILE.exists() else []
topic_info = json.loads(TOPIC_INFO_FILE.read_text(encoding='utf-8')) if TOPIC_INFO_FILE.exists() else []

if TOPIC_ASSIGN_FILE.exists():
    with gzip.open(TOPIC_ASSIGN_FILE, 'rt', encoding='utf-8') as _fh:
        topic_assign = json.load(_fh)
else:
    topic_assign = []

print(f'System 1 (cosine similarity):   {len(s1_docs):>4} candidates')
print(f'System 2 (BERTopic):            {len(s2_docs):>4} candidates')
print(f'System 2 Noise pool (topic -1): {len(noise_docs):>4} docs')

System 1 (cosine similarity):    157 candidates
System 2 (BERTopic):             166 candidates
System 2 Noise pool (topic -1):  298 docs


## §2. Compile union & save

In [5]:
s1_by_title = {d['title'].strip(): d for d in s1_docs}
s2_by_title = {d['title'].strip(): d for d in s2_docs}
all_titles  = sorted(set(s1_by_title) | set(s2_by_title))

union  = []
counts = {'both': 0, 's1_only': 0, 's2_only': 0}

for title in all_titles:
    in_s1  = title in s1_by_title
    in_s2  = title in s2_by_title
    base   = s1_by_title[title] if in_s1 else s2_by_title[title]
    source = 'both' if (in_s1 and in_s2) else ('s1_only' if in_s1 else 's2_only')
    union.append({**base, 'system_sources': source})
    counts[source] += 1

union.sort(key=lambda x: x.get('date_published', ''))
OUT_FILE.write_text(json.dumps(union, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Saved {len(union)} candidates to {OUT_FILE.name}')

df_union = pd.DataFrame(union)
df_union['year'] = pd.to_datetime(df_union['date_published'], errors='coerce').dt.year

Saved 260 candidates to germany_2016-2018_pooled_union.json


## §3. Overview

In [6]:
n_both  = counts['both']
n_s1    = counts['s1_only']
n_s2    = counts['s2_only']
n_total = len(union)

pd.DataFrame({
    'n': [n_both, n_s1, n_s2, n_total],
    'share': [f'{n_both/n_total:.0%}', f'{n_s1/n_total:.0%}', f'{n_s2/n_total:.0%}', '100%'],
}, index=['both systems (high confidence)', 'System 1 only (cosine sim)', 'System 2 only (BERTopic)', 'total'])

,n,share
both systems (high confidence),59,23%
System 1 only (cosine sim),96,37%
System 2 only (BERTopic),105,40%
total,260,100%


In [7]:
pd.crosstab(df_union['year'], df_union['system_sources'], margins=True)

system_sources,both,s1_only,s2_only,All
year,,,,
2016,22,36,37,95
2017,21,39,41,101
2018,16,21,27,64
All,59,96,105,260


## §4. High-confidence candidates (both systems)

Found by both cosine similarity and BERTopic — the strongest candidates.

In [8]:
df_both = df_union[df_union['system_sources'] == 'both'][['title', 'date_published', 'year']].reset_index(drop=True)
print(f'{len(df_both)} docs found by both systems')
df_both

59 docs found by both systems


,title,date_published,year
0,Verordnung über die Prüfung zum anerkannten Fortbildungsabschluss Geprüfter Industriemeister  Fachrichtung Süßwaren...,2016-01-29T00:00:00Z,2016
1,Verordnung über die Prüfung zum anerkannten Fortbildungsabschluss Geprüfter Kaufmännischer Fachwirt nach der Handwer...,2016-03-07T00:00:00Z,2016
2,Verordnung über die Berufsausbildung zum Dachdecker und zur Dachdeckerin (Dachdeckerausbildungsverordnung  DachAusbV),2016-05-02T00:00:00Z,2016
3,Verordnung über die Meisterprüfung in den Teilen I und II im Boots- und Schiffbauer-Handwerk (Bootsbauermeisterveror...,2016-05-02T00:00:00Z,2016
4,Achtundvierzigste Verordnung über das anzurechnende Einkommen nach dem Bundesversorgungsgesetz (Achtundvierzigste An...,2016-06-23T00:00:00Z,2016
5,Gesetz zur Stärkung der beruflichen Weiterbildung und des Versicherungsschutzes in der Arbeitslosenversicherung (Arb...,2016-07-22T00:00:00Z,2016
6,Gesetz zur Weiterentwicklung des Behindertengleichstellungsrechts,2016-07-26T00:00:00Z,2016
7,Verordnung zur Festlegung der der Revision unterliegenden Bundesbeteiligung an den Kosten der Unterkunft und Heizung...,2016-07-26T00:00:00Z,2016
8,Siebte Verordnung zur Änderung der Arbeitslosengeld II/Sozialgeld-Verordnung,2016-07-29T00:00:00Z,2016
9,Künstlersozialabgabe-Verordnung 2017,2016-08-11T00:00:00Z,2016


## §5. System 1 only (cosine similarity)

Found by cosine similarity but not by BERTopic — vocabulary close to seed descriptions
but did not cluster with other welfare-policy laws.

In [9]:
df_s1 = df_union[df_union['system_sources'] == 's1_only'][['title', 'date_published', 'year']].reset_index(drop=True)
print(f'{len(df_s1)} docs found only by cosine similarity')
df_s1

96 docs found only by cosine similarity


,title,date_published,year
0,Berichtigung der Deutsche-Bahn-Schichtzulagenerhöhungsverordnung,2016-01-29T00:00:00Z,2016
1,Gesetz zur Verbesserung der Registrierung und des Datenaustausches zu aufenthalts- und asylrechtlichen Zwecken (Date...,2016-02-04T00:00:00Z,2016
2,Verordnung über den Übergang einer Teilstrecke der Bundeswasserstraße Lühe auf den Unterhaltungsverband Nummer 15 Au...,2016-02-04T00:00:00Z,2016
3,Verordnung über die Bescheinigung über die Meldung als Asylsuchender (Ankunftsnachweisverordnung  AKNV),2016-02-05T00:00:00Z,2016
4,Vierzehnte Verordnung zur Änderung der Arzneimittelverschreibungsverordnung,2016-02-23T00:00:00Z,2016
5,Gesetz zur Einführung beschleunigter Asylverfahren,2016-03-16T00:00:00Z,2016
6,Gesetz zur Umsetzung der Wohnimmobilienkreditrichtlinie und zur Änderung handelsrechtlicher Vorschriften,2016-03-16T00:00:00Z,2016
7,Gesetz zur erleichterten Ausweisung von straffälligen Ausländern und zum erweiterten Ausschluss der Flüchtlingsanerk...,2016-03-16T00:00:00Z,2016
8,"Verordnung über die Anlage des Sicherungsvermögens von Pensionskassen, Sterbekassen und kleinen Versicherungsunterne...",2016-04-21T00:00:00Z,2016
9,Verordnung über die Finanzierung des Sicherungsfonds für die Lebensversicherer (SichLVFinV),2016-04-21T00:00:00Z,2016


### §5a. Decision — System 1 only

Run the triage cell below: it shows the System-1-only policies one at a time and asks for a decision.

`y` = include `n` / Enter = exclude `b` = back `q` = quit

Decisions are saved **by title** to `{OUTPUT_PREFIX}_s1_only_decisions.json` after every answer, so they stay valid if the row order of the §5 table changes. Re-running the cell resumes at the first undecided policy. To revise earlier decisions, set `S1_START_AT` to a position (1-based, as shown in the `[i/n]` counter).

§8 refuses to save while any System-1-only policy is still undecided.

The first cell below is a one-off: it converts the earlier index-based decisions (`S1_INCLUDED_INDICES`) into the decisions file. It does nothing once that file exists.

In [10]:
# ── DECISION: System 1 only — interactive triage ─────────────────────────────
# y = include   n = exclude   Enter = no (or keep the current decision when revising)
# b = back   q = quit (progress is saved)
# decisions are stored by title in S1_DECISIONS_FILE (set in the configuration
# cell), so re-running this cell resumes where you stopped

from IPython.display import clear_output

S1_START_AT = None   # None = first undecided policy; or a position (1-based) to revise from there

s1_decisions = (json.loads(S1_DECISIONS_FILE.read_text(encoding='utf-8'))
                if S1_DECISIONS_FILE.exists() else {})

def _save_s1():
    S1_DECISIONS_FILE.write_text(json.dumps(s1_decisions, ensure_ascii=False, indent=2), encoding='utf-8')

s1_titles = df_s1['title'].str.strip().tolist()
if S1_START_AT is not None:
    i = max(int(S1_START_AT) - 1, 0)
else:
    i = next((k for k, t in enumerate(s1_titles) if t not in s1_decisions), len(s1_titles))

while i < len(s1_titles):
    row   = df_s1.iloc[i]
    n_yes = sum(s1_decisions.get(t) == 'y' for t in s1_titles)
    prev  = s1_decisions.get(s1_titles[i])
    clear_output(wait=True)   # show only the current policy
    print(f"[{i + 1}/{len(s1_titles)}]  ({n_yes} included so far)\n")
    print(f"  {row['title']}")
    print(f"  {str(row['date_published'])[:10]}")
    if prev:
        print(f"\n  current decision: {prev}  (Enter keeps it)")
    try:
        choice = input('y = yes   n / Enter = no   b = back   q = quit  > ').strip().lower()
    except KeyboardInterrupt:   # kernel interrupt = quit
        break
    if choice == 'q':
        break
    if choice == 'b':
        i = max(i - 1, 0)
        continue
    if choice == 'y':
        s1_decisions[s1_titles[i]] = 'y'
    elif choice == 'n' or prev is None:
        s1_decisions[s1_titles[i]] = 'n'
    _save_s1()
    i += 1

_save_s1()
S1_INCLUDED_TITLES = {t for t in s1_titles if s1_decisions.get(t) == 'y'}
n_s1_open = sum(t not in s1_decisions for t in s1_titles)
print(f"\n{len(S1_INCLUDED_TITLES)} included, {len(s1_titles) - len(S1_INCLUDED_TITLES) - n_s1_open} excluded, "
      f"{n_s1_open} still undecided" + ("  ← re-run this cell to continue" if n_s1_open else ""))

[96/96]  (46 included so far)

  Abschlusshinweis: für Bundesgesetzblatt Teil I und Teil II
  2018-12-31

46 included, 50 excluded, 0 still undecided


## §6. System 2 only (BERTopic)

Found by BERTopic but not by cosine similarity — likely welfare-policy laws with
unusual vocabulary that did not match the seed descriptions. Inspect topic by topic.

Topic metadata is joined from `_berttopic_topic_assignments.json` since the
candidates file contains only raw documents without topic labels.

In [11]:
title_to_topic = {entry['doc']['title'].strip(): entry['topic_id'] for entry in topic_assign}
topic_to_words = {t['topic_id']: t['top_words'] for t in topic_info}

df_s2 = df_union[df_union['system_sources'] == 's2_only'].copy()
df_s2['berttopic_topic'] = df_s2['title'].str.strip().map(title_to_topic)
print(f'{len(df_s2)} docs found only by BERTopic\n')

for topic_id, group in df_s2.groupby('berttopic_topic'):
    words = topic_to_words.get(topic_id, [])
    top_words_str = ', '.join(words[:6])
    print(f'── topic {int(topic_id)}  (n={len(group)})  top words: {top_words_str}')
    display(group[['title', 'date_published']].reset_index(drop=True))
    print()

105 docs found only by BERTopic

── topic 1  (n=70)  top words: fortbildungsabschluss geprüfter, geprüfter, industriemeister, fachrichtung, ausbildungsverordnung berufsausbildung, fachkraft


,title,date_published
0,Verordnung zur Änderung der Verordnung über den Aufstieg in den höheren nichttechnischen Verwaltungsdienst des Bunde...,2016-01-11T00:00:00Z
1,Erste Verordnung zur Änderung der Verordnung über die Berufsausbildung zum Bäcker/zur Bäckerin,2016-02-15T00:00:00Z
2,Verordnung zur Änderung und Aufhebung von Fortbildungs- und Umschulungsverordnungen,2016-03-07T00:00:00Z
3,Verordnung über die Berufsausbildung zum Fischwirt und zur Fischwirtin (Fischwirtausbildungsverordnung  FischwAusbV),2016-03-07T00:00:00Z
4,Gesetz zur Änderung des Hochschulstatistikgesetzes,2016-03-10T00:00:00Z
5,Neufassung der Trinkwasserverordnung,2016-03-16T00:00:00Z
6,Drittes Gesetz zur Änderung des Aufstiegsfortbildungsförderungsgesetzes,2016-04-08T00:00:00Z
7,"Verordnung über die Berufsausbildung zum Anlagenmechaniker für Sanitär-, Heizungs- und Klimatechnik und zur Anlagenm...",2016-05-02T00:00:00Z
8,Verordnung über die Berufsausbildung zum Hörakustiker und zur Hörakustikerin (Hörakustikerausbildungsverordnung  Hö...,2016-05-02T00:00:00Z
9,Verordnung über die Berufsausbildung zum Rollladen- und Sonnenschutzmechatroniker und zur Rollladen- und Sonnenschut...,2016-05-09T00:00:00Z



── topic 8  (n=6)  top words: behindertengleichstellungsrecht bereinigung, auslandstrennungsgeldrecht, auswärtig, ausschluss verfassungsfeindlich, auslandsumzugskostenverordnung europawahlordnung, beförderungsämter bundesunmittelbar


,title,date_published
0,Verordnung über die für Staatsangehörige eines Mitgliedstaates der Europäischen Union oder eines anderen Vertragssta...,2016-03-31T00:00:00Z
1,Neuntes Gesetz zur Änderung des Zweiten Buches Sozialgesetzbuch  Rechtsvereinfachung  sowie zur vorübergehenden Au...,2016-07-29T00:00:00Z
2,Berichtigung des Neunten Gesetzes zur Änderung des Zweiten Buches Sozialgesetzbuch  Rechtsvereinfachung  sowie zur...,2016-12-02T00:00:00Z
3,"Verordnung zur Änderung der Laufbahn-, Ausbildungs- und Prüfungsverordnungen für den Auswärtigen Dienst",2016-12-13T00:00:00Z
4,Gesetz zur Weiterentwicklung der Versorgung und der Vergütung für psychiatrische und psychosomatische Leistungen (Ps...,2016-12-23T00:00:00Z
5,Gesetz zur Änderung des Beamtenstatusgesetzes und des Bundesbeamtengesetzes sowie weiterer dienstrechtlicher Vorschr...,2018-12-06T00:00:00Z



── topic 9  (n=3)  top words: finanzausgleichsgesetz, finanzausgleichsgesetz ausgleichsjahr, ausgleichsjahr festsetzung, gewerbesteuerumlage gemeindefinanzreformgesetz, festsetzung umlagesatz, gemeindefinanzreformgesetz


...

,title,date_published
0,Gesetz zur Reform der Investmentbesteuerung (Investmentsteuerreformgesetz  InvStRefG),2016-07-26T00:00:00Z
1,Gesetz zur Anpassung des Erbschaftsteuer- und Schenkungsteuergesetzes an die Rechtsprechung des Bundesverfassungsger...,2016-11-09T00:00:00Z



── topic 19  (n=8)  top words: frist sozialgesetzbuch, sozialgesetzbuch künstlersozialabgabe, bundesagentur arbeit, arbeit weiterleitung, anspruch ausländisch, arbeitsschutzzuständige


,title,date_published
0,Verordnung zur Verlängerung der Frist nach § 291 Absatz 2b Satz 6 des Fünften Buches Sozialgesetzbuch,2016-12-02T00:00:00Z
1,Dritte Verordnung zur Änderung der Verordnung über den Beirat und die Ausschüsse bei der Künstlersozialkasse,2017-01-26T00:00:00Z
2,Erste Verordnung zur Änderung der Akkreditierungs- und Zulassungsverordnung Arbeitsförderung,2017-02-02T00:00:00Z
3,Zweite Verordnung zur Änderung der Verwaltungskostenfeststellungsverordnung,2017-07-12T00:00:00Z
4,Verordnung zur Verlängerung der Frist nach § 291 Absatz 2b Satz 14 des Fünften Buches Sozialgesetzbuch,2017-11-24T00:00:00Z
5,Anordnung des Vorstands der Bundesagentur für Arbeit über die Übertragung von Befugnissen auf dem Gebiet des Beamten...,2018-01-19T00:00:00Z
6,Bekanntmachung über das Inkrafttreten des Artikels 2 Nummer 1 Buchstabe e und Nummer 8 des Gesetzes zur Steigerung d...,2018-04-30T00:00:00Z
7,Verordnung über die Verpflichtung der Bundesagentur für Arbeit zur Weiterleitung von Betriebsdaten an die für den Ar...,2018-04-30T00:00:00Z



── topic 20  (n=6)  top words: rentenversicherung alterssicherung, landwirt rentenwertbestimmungsverordnung, sozialversicherungsentgeltverordnung, sozialversicherungs rechengrößenverordnung, sozialversicherung sozialversicherungs, rechengrößenverordnung sozialversicherungsentgeltverordnung


,title,date_published
0,Verordnung zur Bestimmung der Rentenwerte in der gesetzlichen Rentenversicherung und in der Alterssicherung der Land...,2016-06-23T00:00:00Z
1,Sechstes Gesetz zur Änderung des Vierten Buches Sozialgesetzbuch und anderer Gesetze (6. SGB IV-Änderungsgesetz  6....,2016-11-16T00:00:00Z
2,Zweite Verordnung zur Änderung der Verordnung zur Durchführung des § 90 Abs. 2 Nr. 9 des Zwölften Buches Sozialgeset...,2017-03-29T00:00:00Z
3,Verordnung zur Bestimmung der Rentenwerte in der gesetzlichen Rentenversicherung und in der Alterssicherung der Land...,2017-06-13T00:00:00Z
4,Erste Verordnung zur Änderung der RfB-Verordnung,2017-07-31T00:00:00Z
5,Verordnung zur Bestimmung der Rentenwerte in der gesetzlichen Rentenversicherung und in der Alterssicherung der Land...,2018-06-14T00:00:00Z


### §6a. Decision — System 2 only (topic level)

Review the topic groups above. Enter the **topic IDs** you want to keep in `S2_KEPT_TOPICS` below — all documents in those topics are included, all others are excluded.  
Leave as `[]` to exclude all System-2-only policies.

In [12]:
# ── DECISION: System 2 only ───────────────────────────────────────────────────
# BERTopic topic IDs to keep in full (all documents in those topics are included).
# Topic IDs are shown in the "── topic N" headers above.
# Example: S2_KEPT_TOPICS = [3, 7, 12]
# Leave as [] to exclude all System-2-only policies.

S2_KEPT_TOPICS = [1, 8, 9, 14, 19, 20]   # ← edit this list

# Individual picks from topics NOT fully kept above.
# Keys: topic ID (from §6 "── topic N" header).
# Values: 0-based row indices within that topic's table (left-most column in §6 display).
# Example: S2_INDIVIDUAL_PICKS = {5: [0, 3], 14: [2]}

S2_INDIVIDUAL_PICKS = {}   # ← edit this dict

## §7. Noise pool spot-check

Topic -1: documents HDBSCAN could not assign to any cluster. Spot-check a sample
to verify they are correctly excluded as non-welfare-policy laws.

In [13]:
SAMPLE_N = 30
if noise_docs:
    sample   = random.sample(noise_docs, min(SAMPLE_N, len(noise_docs)))
    df_noise = pd.DataFrame(sample)
    cols     = [c for c in ['title', 'date_published'] if c in df_noise.columns]
    df_noise = df_noise[cols].sort_values('date_published').reset_index(drop=True)
    print(f'Random sample of {len(df_noise)} from {len(noise_docs)} noise docs (topic -1)')
    display(df_noise)
else:
    print('No noise pool file found.')

Random sample of 30 from 298 noise docs (topic -1)


,title,date_published
0,Gesetz zur Durchführung der Verordnung (EU) Nr. 1007/2011 und zur Ablösung des Textilkennzeichnungsgesetzes,2016-02-23T00:00:00Z
1,Gesetz zur Umsetzung der Richtlinie über alternative Streitbeilegung in Verbraucherangelegenheiten und zur Durchführ...,2016-02-25T00:00:00Z
2,Berichtigung der Zweiten Verordnung zur Änderung von Vorschriften über die Berufsausbildung in der Landwirtschaft,2016-03-07T00:00:00Z
3,Verordnung zur Bereinigung quotenrechtlicher Vorschriften und zur Umsetzung europarechtlicher Vorgaben zur Treibhaus...,2016-04-08T00:00:00Z
4,Verordnung zur Bestimmung Kritischer Infrastrukturen nach dem BSI-Gesetz (BSI-Kritisverordnung  BSI-KritisV),2016-05-02T00:00:00Z
5,"Bekanntmachung der Änderung der zur Entgegennahme von Patent-, Gebrauchsmuster-, Marken- und Designanmeldungen befug...",2016-05-09T00:00:00Z
6,Zweiundzwanzigste Verordnung zur Anpassung des Bemessungsbetrages und von Geldleistungen nach dem Bundesversorgungsg...,2016-06-23T00:00:00Z
7,Erstes Gesetz zur Änderung des Direktzahlungen-Durchführungsgesetzes,2016-10-27T00:00:00Z
8,Gesetz zur Neuregelung des Mikrozensus und zur Änderung weiterer Statistikgesetze,2016-12-13T00:00:00Z
9,Dritte Verordnung zur Änderung der Binnenschiffsuntersuchungsordnung und sonstiger schifffahrtsrechtlicher Vorschriften,2016-12-22T00:00:00Z


---

*Add further inspection cells below.*

## §8. Final policy set — compile & save

Applies the decisions from §5a and §6a.  
All "both" candidates are always included; System-1-only and System-2-only policies are filtered based on your selections above.

In [14]:
# ── Apply decisions ───────────────────────────────────────────────────────────
# System-1-only decisions are re-read from the decisions file (§5a), so this cell
# does not depend on the triage cell having run in the current kernel session.
s1_decisions = (json.loads(S1_DECISIONS_FILE.read_text(encoding='utf-8'))
                if S1_DECISIONS_FILE.exists() else {})
_s1_titles   = df_s1['title'].str.strip()
n_s1_open    = int((~_s1_titles.isin(s1_decisions)).sum())
if n_s1_open:
    raise RuntimeError(f'{n_s1_open} System-1-only policies are still undecided — finish the §5a triage first.')
S1_INCLUDED_TITLES = {t for t in _s1_titles if s1_decisions.get(t) == 'y'}

df_both_docs   = df_union[df_union['system_sources'] == 'both'].copy()
df_s1_selected = df_s1[_s1_titles.isin(S1_INCLUDED_TITLES)].copy()
df_s2_selected = (
    df_s2[df_s2['berttopic_topic'].isin(S2_KEPT_TOPICS)].copy()
    if S2_KEPT_TOPICS else df_s2.iloc[[]]
)

# Individual picks from topics not fully kept
_pick_frames = []
for _topic_id, _indices in S2_INDIVIDUAL_PICKS.items():
    if _topic_id in S2_KEPT_TOPICS:
        continue  # already fully included
    _group = df_s2[df_s2['berttopic_topic'] == _topic_id].reset_index(drop=True)
    _pick_frames.append(_group.iloc[_indices])
df_s2_individual = pd.concat(_pick_frames, ignore_index=True) if _pick_frames else df_s2.iloc[[]]

keep_titles = (
    set(df_both_docs['title'].str.strip())
    | set(df_s1_selected['title'].str.strip())
    | set(df_s2_selected['title'].str.strip())
    | set(df_s2_individual['title'].str.strip())
)

final_records = [r for r in union if r['title'].strip() in keep_titles]
final_records.sort(key=lambda x: x.get('date_published', ''))

FINAL_FILE = PROJECT_ROOT / 'data' / 'processed' / 'germany' / f'{OUTPUT_PREFIX}_final_policy_set.json'
FINAL_FILE.write_text(json.dumps(final_records, ensure_ascii=False, indent=2), encoding='utf-8')

n_both_final       = len(df_both_docs)
n_s1_final         = len(df_s1_selected)
n_s2_final         = len(df_s2_selected)
n_s2_individual    = len(df_s2_individual)
n_total_final      = len(final_records)

print(f'Accepted from both systems:           {n_both_final:>4}')
print(f'Accepted from System 1 only:          {n_s1_final:>4}  (of {n_s1} candidates)')
print(f'Accepted from System 2 only (topics): {n_s2_final:>4}  (of {n_s2} candidates, {len(S2_KEPT_TOPICS)} topic(s) kept)')
print(f'Accepted from System 2 (individual):  {n_s2_individual:>4}  (across {len(S2_INDIVIDUAL_PICKS)} topic(s))')
print(f'─────────────────────────────────────────────')
print(f'Final policy set (total):             {n_total_final:>4}')
print(f'Saved to: {FINAL_FILE.name}')

Accepted from both systems:             59
Accepted from System 1 only:            46  (of 96 candidates)
Accepted from System 2 only (topics):   95  (of 105 candidates, 6 topic(s) kept)
Accepted from System 2 (individual):     0  (across 0 topic(s))
─────────────────────────────────────────────
Final policy set (total):              200
Saved to: germany_2016-2018_final_policy_set.json


## §9. Export evaluation log

Generates a dated Markdown log documenting all inputs, counts, decisions, and outputs.  
**Run after §8.**

In [16]:
import datetime

today = datetime.date.today().isoformat()
log_path = BASE / f'pooled_eval_log_{OUTPUT_PREFIX}_{today}.md'

s2_topics_detail = ', '.join(
    f'topic {t} ({", ".join(str(w) for w in topic_to_words.get(t, [])[:5])})'
    for t in sorted(S2_KEPT_TOPICS)
) if S2_KEPT_TOPICS else '*(none)*'

log_md = f"""# Pooled Union Evaluation Log
Generated: {today}

## 1. Input files

| role | file |
|---|---|
| System 1 (cosine similarity) | `{S1_FILE.name}` |
| System 2 (BERTopic candidates) | `{S2_FILE.name}` |
| System 2 noise pool | `{NOISE_FILE.name}` |
| BERTopic topic assignments | `{TOPIC_ASSIGN_FILE.name}` |
| BERTopic topic info | `{TOPIC_INFO_FILE.name}` |

## 2. §1 — Candidate counts (raw input)

| source | n |
|---|---|
| System 1 (cosine similarity) | {len(s1_docs)} |
| System 2 (BERTopic) | {len(s2_docs)} |
| System 2 noise pool (topic -1) | {len(noise_docs)} |

## 3. §3 — Pooled union overview

| group | n | share |
|---|---|---|
| Both systems (high confidence) | {n_both} | {n_both/n_total:.0%} |
| System 1 only (cosine sim) | {n_s1} | {n_s1/n_total:.0%} |
| System 2 only (BERTopic) | {n_s2} | {n_s2/n_total:.0%} |
| **Total** | **{n_total}** | **100%** |

## 4. §5a — System 1 only decision (interactive triage)

- Candidates reviewed: {n_s1}
- Decisions file (by title): `{S1_DECISIONS_FILE.name}`
- Still undecided: {n_s1_open}
- **Policies included: {n_s1_final}**

## 5. §6a — System 2 only decision (topic level)

- Candidates reviewed: {n_s2} across {df_s2['berttopic_topic'].nunique()} topics
- Topics kept: {len(S2_KEPT_TOPICS)} — {s2_topics_detail}
- **Policies included: {n_s2_final}**

## 6. §8 — Final policy set

| group | n |
|---|---|
| Accepted by both systems | {n_both_final} |
| Accepted from System 1 only | {n_s1_final} |
| Accepted from System 2 only | {n_s2_final} |
| **Final total** | **{n_total_final}** |

## 7. Output file

`{FINAL_FILE.name}`
"""

log_path.write_text(log_md, encoding='utf-8')
print(f'Log saved to: {log_path.name}')

Log saved to: pooled_eval_log_germany_2016-2018_2026-10-29.md
